In [1]:
import pandas as pd
import numpy as np
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, r2_score
from scikeras.wrappers import KerasRegressor
from sklearn.compose import TransformedTargetRegressor 
import joblib

In [2]:
df=pd.read_csv('housing.csv')
df

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY
...,...,...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,78100.0,INLAND
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,77100.0,INLAND
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,92300.0,INLAND
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,84700.0,INLAND


In [3]:
df.isnull().sum()

longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

In [4]:
df=df.dropna()

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
X = df.drop("median_house_value", axis=1)
y = df["median_house_value"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_features = X.select_dtypes(include=['float64', 'int64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaled = y_scaler.transform(y_test.values.reshape(-1, 1))

n_inputs = X_train_processed.shape[1] 

model = keras.Sequential([
    keras.layers.Dense(128, activation='relu', input_shape=[n_inputs]),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(1, activation='linear')
])

model.compile(
    optimizer='adam',
    loss='mean_squared_error'
)

print("Starting training...")
history = model.fit(
    X_train_processed,
    y_train_scaled,  
    epochs=40,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)
print("Training finished.")

y_pred_scaled = model.predict(X_test_processed)

y_pred = y_scaler.inverse_transform(y_pred_scaled)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n--- Evaluation (in Dollars) ---")
print(f"RMSE: ${rmse:,.2f}")
print(f"R-squared (R^2): {r2:.4f}")

Starting training...
Epoch 1/40


c:\anaconda\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.3209 - val_loss: 0.2932
Epoch 2/40
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.2592 - val_loss: 0.2778
Epoch 3/40
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.2436 - val_loss: 0.2590
Epoch 4/40
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.2346 - val_loss: 0.2548
Epoch 5/40
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.2255 - val_loss: 0.2465
Epoch 6/40
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.2209 - val_loss: 0.2448
Epoch 7/40
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.2158 - val_loss: 0.2510
Epoch 8/40
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.2130 - val_loss: 0.2434
Epoch 9/40
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.2066 - val_loss: 0.2352
Epoch 10/40
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.2045 - val_loss: 0.2433
Epoch 11/40
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.2018 - val_loss: 0.2390
Epoch 12/40
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.

In [ ]:
joblib.dump(preprocessor, 'preprocessor.joblib')

In [ ]:
joblib.dump(y_scaler, 'y_scaler.joblib')

In [ ]:
model.save('housing_model.keras')